In [1]:
# 📦 1. Importações iniciais
from yt_dlp import YoutubeDL
from pathlib import Path
import json

# 🛠️ 2. Parâmetros configuráveis
URL = "https://www.youtube.com/watch?v=f-tmb-Tcn_k"

# Estrutura de pastas organizada
VIDEO_DIR = Path("data/raw/videos")
AUDIO_DIR = Path("data/raw/audio") 
META_DIR = Path("data/raw/metadata")

# Criar todas as pastas
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

print("📁 Pastas criadas com sucesso!")


📁 Pastas criadas com sucesso!


In [2]:
# 🎬 3. Download do VÍDEO
print("🔄 Fazendo download do vídeo...")

video_path = VIDEO_DIR / "rpg_video.%(ext)s"

video_opts = {
    "format": "bestvideo+bestaudio/best",  # Melhor qualidade disponível
    "outtmpl": str(video_path),
    "writeinfojson": True,  # Salva metadados em .info.json
}

try:
    with YoutubeDL(video_opts) as ydl:
        info = ydl.extract_info(URL, download=True)
    
    print("✅ Vídeo baixado com sucesso!")
    print(f"📹 Título: {info.get('title', 'Sem título')}")
    print(f"⏱️ Duração: {info.get('duration', 0)//60} minutos")
    print(f"📺 Canal: {info.get('uploader', 'Desconhecido')}")
    
except Exception as e:
    print(f"❌ Erro no download do vídeo: {e}")
    print("⚠️ Verifique a URL ou sua conexão de internet")

🔄 Fazendo download do vídeo...
[youtube] Extracting URL: https://www.youtube.com/watch?v=f-tmb-Tcn_k
[youtube] f-tmb-Tcn_k: Downloading webpage
[youtube] f-tmb-Tcn_k: Downloading ios player API JSON
[youtube] f-tmb-Tcn_k: Downloading mweb player API JSON
[youtube] f-tmb-Tcn_k: Downloading player 6e20d3a8


[youtube] f-tmb-Tcn_k: Downloading m3u8 information
[info] f-tmb-Tcn_k: Downloading 1 format(s): 616+234
[info] Writing video metadata as JSON to: data/raw/videos/rpg_video.info.json
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1676
[download] Destination: data/raw/videos/rpg_video.f616.mp4
[download] 100% of    1.67GiB in 00:11:46 at 2.43MiB/s                     
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1676
[download] Destination: data/raw/videos/rpg_video.f234.mp4
[download] 100% of  144.01MiB in 00:02:05 at 1.15MiB/s                     
[Merger] Merging formats into "data/raw/videos/rpg_video.mp4"
Deleting original file data/raw/videos/rpg_video.f616.mp4 (pass -k to keep)
Deleting original file data/raw/videos/rpg_video.f234.mp4 (pass -k to keep)
✅ Vídeo baixado com sucesso!
📹 Título: #FabulaUltima: OS CHAMPIÕES — Episódio #01 | Campanha Oficial Brasileira
⏱️ Duração: 154 minutos
📺 Canal: Jambô Editora


In [ ]:
# 🔊 4. Download do ÁUDIO separadamente
print("\n🔄 Fazendo download do áudio...")

audio_path = AUDIO_DIR / "rpg_audio.%(ext)s"

audio_opts = {
    "format": "bestaudio/best",  # Apenas áudio, melhor qualidade
    "outtmpl": str(audio_path),
    "postprocessors": [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',  # Converter para WAV
        'preferredquality': '192',
    }],
    "prefer_ffmpeg": True,  # ← ESTA LINHA
}

try:
    with YoutubeDL(audio_opts) as ydl:
        ydl.download([URL])
    
    print("✅ Áudio baixado e convertido para WAV!")
    
    # Verificar se o arquivo foi criado
    audio_files = list(AUDIO_DIR.glob("rpg_audio.*"))
    if audio_files:
        print(f"📄 Arquivo de áudio: {audio_files[0].name}")
    
except Exception as e:
    print(f"❌ Erro no download do áudio: {e}")
    print("⚠️ Certifique-se de que o ffmpeg está instalado")


🔄 Fazendo download do áudio...
[youtube] Extracting URL: https://www.youtube.com/watch?v=f-tmb-Tcn_k
[youtube] f-tmb-Tcn_k: Downloading webpage
[youtube] f-tmb-Tcn_k: Downloading ios player API JSON
[youtube] f-tmb-Tcn_k: Downloading mweb player API JSON
[youtube] f-tmb-Tcn_k: Downloading player 6e20d3a8


[youtube] f-tmb-Tcn_k: Downloading m3u8 information
[info] f-tmb-Tcn_k: Downloading 1 format(s): 234
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1676
[download] Destination: data/raw/audio/rpg_audio.mp4
[download] 100% of  144.01MiB in 00:01:04 at 2.24MiB/s                     
[ExtractAudio] Destination: data/raw/audio/rpg_audio.wav
Deleting original file data/raw/audio/rpg_audio.mp4 (pass -k to keep)
✅ Áudio baixado e convertido para WAV!
📄 Arquivo de áudio: rpg_audio.wav


In [4]:
# 📊 5. Extrair METADADOS detalhados
print("\n🔄 Extraindo metadados detalhados...")

try:
    with YoutubeDL({'quiet': True}) as ydl:
        info = ydl.extract_info(URL, download=False)  # Só extrai info, não baixa
    
    # Organizar metadados importantes
    metadata = {
        "titulo": info.get('title', ''),
        "duracao_segundos": info.get('duration', 0),
        "duracao_legivel": f"{info.get('duration', 0)//3600:02d}:{(info.get('duration', 0)%3600)//60:02d}:{info.get('duration', 0)%60:02d}",
        "canal": info.get('uploader', ''),
        "data_upload": info.get('upload_date', ''),
        "visualizacoes": info.get('view_count', 0),
        "likes": info.get('like_count', 0),
        "descricao": info.get('description', ''),
        "tags": info.get('tags', []),
        "url_original": URL,
        "id_video": info.get('id', ''),
    }
    
    # Salvar metadados em JSON
    metadata_file = META_DIR / "rpg_metadata.json"
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print("✅ Metadados extraídos e salvos!")
    print(f"📋 Arquivo: {metadata_file}")
    
    # Mostrar alguns metadados importantes
    print(f"\n📊 INFORMAÇÕES DO VÍDEO:")
    print(f"   🎬 {metadata['titulo']}")
    print(f"   ⏱️ Duração: {metadata['duracao_legivel']}")
    print(f"   👀 Visualizações: {metadata['visualizacoes']:,}")
    print(f"   👍 Likes: {metadata['likes']:,}")
    print(f"   🏷️ Tags: {len(metadata['tags'])} encontradas")
    
except Exception as e:
    print(f"❌ Erro ao extrair metadados: {e}")


🔄 Extraindo metadados detalhados...


✅ Metadados extraídos e salvos!
📋 Arquivo: data/raw/metadata/rpg_metadata.json

📊 INFORMAÇÕES DO VÍDEO:
   🎬 #FabulaUltima: OS CHAMPIÕES — Episódio #01 | Campanha Oficial Brasileira
   ⏱️ Duração: 02:34:16
   👀 Visualizações: 4,726
   👍 Likes: 427
   🏷️ Tags: 6 encontradas


In [5]:
# 📁 6. Verificação final dos arquivos
print("\n📁 ARQUIVOS GERADOS:")

# Verificar vídeo
video_files = list(VIDEO_DIR.glob("rpg_video.*"))
if video_files:
    for file in video_files:
        if not file.name.endswith('.json'):  # Ignorar arquivo de metadados
            size_mb = file.stat().st_size / (1024*1024)
            print(f"   📹 Vídeo: {file.name} ({size_mb:.1f} MB)")

# Verificar áudio  
audio_files = list(AUDIO_DIR.glob("rpg_audio.*"))
if audio_files:
    for file in audio_files:
        size_mb = file.stat().st_size / (1024*1024)
        print(f"   🔊 Áudio: {file.name} ({size_mb:.1f} MB)")

# Verificar metadados
meta_files = list(META_DIR.glob("*.json"))
if meta_files:
    for file in meta_files:
        print(f"   📋 Metadados: {file.name}")

print("\n🎉 Processo finalizado! Todos os arquivos estão organizados nas pastas.")


📁 ARQUIVOS GERADOS:
   📹 Vídeo: rpg_video.mp4 (1861.5 MB)
   🔊 Áudio: rpg_audio.wav (1557.1 MB)
   📋 Metadados: rpg_metadata.json

🎉 Processo finalizado! Todos os arquivos estão organizados nas pastas.


In [7]:
# 📦 Importações
from yt_dlp import YoutubeDL
from pathlib import Path
from datetime import datetime
import json
import re

def get_next_episode_number(campanha_dir: Path) -> str:
    """
    Verifica quantos episódios já existem e retorna o próximo número
    """
    video_dir = campanha_dir / "videos"
    
    if not video_dir.exists():
        return "ep01"  # Primeiro episódio
    
    # Buscar todos os arquivos que começam com "ep"
    existing_files = list(video_dir.glob("ep*"))
    
    if not existing_files:
        return "ep01"  # Nenhum episódio encontrado
    
    # Extrair números dos episódios existentes
    episode_numbers = []
    for file in existing_files:
        # Procurar padrão "ep" seguido de números
        match = re.search(r'ep(\d+)', file.name)
        if match:
            episode_numbers.append(int(match.group(1)))
    
    if not episode_numbers:
        return "ep01"
    
    # Próximo número
    next_number = max(episode_numbers) + 1
    return f"ep{next_number:02d}"  # Formato ep01, ep02, etc.

def get_campaign_from_url_or_title(url: str, title: str = "") -> str:
    """
    Tenta identificar a campanha pela URL ou título
    """
    # Dicionário de campanhas conhecidas
    known_campaigns = {
        "fabula ultima": "fabula_ultima",
        "ordem paranormal": "ordem_paranormal", 
        "critical role": "critical_role",
        "jambô": "jambo_editora",
        "cellbit": "ordem_paranormal"
    }
    
    # Verificar URL primeiro
    url_lower = url.lower()
    for keyword, campaign in known_campaigns.items():
        if keyword in url_lower:
            return campaign
    
    # Verificar título se fornecido
    if title:
        title_lower = title.lower()
        for keyword, campaign in known_campaigns.items():
            if keyword in title_lower:
                return campaign
    
    # Fallback: perguntar ao usuário
    return None

def ask_user_for_campaign() -> str:
    """
    Pergunta ao usuário qual campanha quando não consegue detectar automaticamente
    """
    print("\n🤔 Não consegui identificar a campanha automaticamente.")
    print("Campanhas disponíveis:")
    print("  1. fabula_ultima")
    print("  2. ordem_paranormal") 
    print("  3. critical_role")
    print("  4. nova_campanha (criar nova)")
    
    while True:
        choice = input("Digite o número ou nome da campanha: ").strip()
        
        if choice == "1":
            return "fabula_ultima"
        elif choice == "2":
            return "ordem_paranormal"
        elif choice == "3":
            return "critical_role"
        elif choice == "4":
            new_name = input("Nome da nova campanha (sem espaços): ").strip()
            return new_name.replace(" ", "_").lower()
        elif choice.replace(" ", "_").lower():
            return choice.replace(" ", "_").lower()
        else:
            print("❌ Opção inválida. Tente novamente.")

# 🛠️ Configuração
URL = "https://youtu.be/D-CHQSm1YwU"

# Data de extração
DATA_EXTRACAO = datetime.now().strftime("%Y%m%d")

print("🔍 Analisando vídeo para detectar campanha e episódio...")

# Primeiro, extrair informações do vídeo
try:
    with YoutubeDL({'quiet': True}) as ydl:
        info = ydl.extract_info(URL, download=False)
    
    titulo = info.get('title', '')
    video_id = info.get('id', 'unknown')
    upload_date = info.get('upload_date', '00000000')
    
    print(f"📺 Título: {titulo}")
    
    # Tentar detectar campanha
    campanha = get_campaign_from_url_or_title(URL, titulo)
    
    if not campanha:
        campanha = ask_user_for_campaign()
    
    print(f"🎯 Campanha detectada: {campanha}")
    
except Exception as e:
    print(f"❌ Erro ao extrair informações: {e}")
    campanha = ask_user_for_campaign()
    video_id = 'unknown'
    upload_date = '00000000'
    titulo = 'Título desconhecido'

# Estrutura de pastas
BASE_DIR = Path("data/raw")
CAMPANHA_DIR = BASE_DIR / campanha

# Detectar próximo episódio automaticamente
episodio = get_next_episode_number(CAMPANHA_DIR)
print(f"📍 Próximo episódio: {episodio}")

# Criar estrutura de pastas
VIDEO_DIR = CAMPANHA_DIR / "videos"
AUDIO_DIR = CAMPANHA_DIR / "audio"
META_DIR = CAMPANHA_DIR / "metadata"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Função para criar nomes de arquivo (versão simplificada)
def create_filename(episodio: str, video_id: str, extension: str) -> str:
    def clean_filename(text: str) -> str:
        cleaned = re.sub(r'[^\w\-]', '', text)
        return cleaned[:15]  # Limita tamanho
    
    episodio_clean = clean_filename(episodio)
    video_id_clean = clean_filename(video_id)
    
    return f"{episodio_clean}_{video_id_clean}.{extension}"

# Nome base do arquivo (simplificado)
base_filename = create_filename(
    episodio=episodio,
    video_id=video_id,
    extension=""
)

print(f"📝 Nome do arquivo: {base_filename}")

# 🎬 Download do vídeo
print("\n🔄 Fazendo download do vídeo...")

video_filename = base_filename + ".%(ext)s"
video_path = VIDEO_DIR / video_filename

video_opts = {
    "format": "bestvideo+bestaudio/best",
    "outtmpl": str(video_path),
    "writeinfojson": True,
}

try:
    with YoutubeDL(video_opts) as ydl:
        ydl.download([URL])
    print(f"✅ Vídeo salvo: {video_filename}")
    
except Exception as e:
    print(f"❌ Erro no download do vídeo: {e}")

# 🔊 Download do áudio
print("\n🔄 Fazendo download do áudio...")

audio_filename = base_filename + ".wav"
audio_path = AUDIO_DIR / audio_filename

audio_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(AUDIO_DIR / (base_filename + ".%(ext)s")),
    "postprocessors": [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '192',
    }],
    "prefer_ffmpeg": True,
}

try:
    with YoutubeDL(audio_opts) as ydl:
        ydl.download([URL])
    print(f"✅ Áudio salvo: {audio_filename}")
    
except Exception as e:
    print(f"❌ Erro no download do áudio: {e}")

# 📊 Salvar metadados
print("\n🔄 Salvando metadados...")

metadata_filename = base_filename + ".json"
metadata_path = META_DIR / metadata_filename

metadata = {
    "campanha": campanha,
    "episodio": episodio,
    "data_extracao": DATA_EXTRACAO,
    "titulo": titulo,
    "url_original": URL,
    "video_id": video_id,
    "upload_date": upload_date,
    "deteccao_automatica": True,
    "arquivos_gerados": {
        "video": video_filename,
        "audio": audio_filename,
        "metadata": metadata_filename
    }
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadados salvos: {metadata_filename}")

# 📋 Relatório final
print(f"\n📊 RESUMO:")
print(f"   📂 Campanha: {campanha}")
print(f"   📍 Episódio: {episodio} (detectado automaticamente)")
print(f"   📹 Vídeo: {video_filename}")
print(f"   🔊 Áudio: {audio_filename}")
print(f"   📋 Metadados: {metadata_filename}")

# Mostrar estrutura de pastas atual
print(f"\n📁 Estrutura atual da campanha:")
for item in sorted(CAMPANHA_DIR.rglob("*")):
    if item.is_file():
        relative_path = item.relative_to(CAMPANHA_DIR)
        print(f"   📄 {relative_path}")

print("\n🎉 Download concluído com numeração automática!")

🔍 Analisando vídeo para detectar campanha e episódio...


📺 Título: #FabulaUltima: OS CHAMPIÕES — Episódio #02 | Campanha Oficial Brasileira

🤔 Não consegui identificar a campanha automaticamente.
Campanhas disponíveis:
  1. fabula_ultima
  2. ordem_paranormal
  3. critical_role
  4. nova_campanha (criar nova)
🎯 Campanha detectada: fabula_ultima
📍 Próximo episódio: ep02
📝 Nome do arquivo: ep02_D-CHQSm1YwU.

🔄 Fazendo download do vídeo...
[youtube] Extracting URL: https://youtu.be/D-CHQSm1YwU
[youtube] D-CHQSm1YwU: Downloading webpage
[youtube] D-CHQSm1YwU: Downloading ios player API JSON
[youtube] D-CHQSm1YwU: Downloading mweb player API JSON
[youtube] D-CHQSm1YwU: Downloading player 69b31e11


[youtube] D-CHQSm1YwU: Downloading m3u8 information
[info] D-CHQSm1YwU: Downloading 1 format(s): 616+234
[info] Writing video metadata as JSON to: data/raw/fabula_ultima/videos/ep02_D-CHQSm1YwU..info.json
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1761
[download] Destination: data/raw/fabula_ultima/videos/ep02_D-CHQSm1YwU..f616.mp4
[download] 100% of    1.73GiB in 00:03:41 at 8.02MiB/s                     
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1761
[download] Destination: data/raw/fabula_ultima/videos/ep02_D-CHQSm1YwU..f234.mp4
[download] 100% of  149.53MiB in 00:02:29 at 1023.60KiB/s                   
[Merger] Merging formats into "data/raw/fabula_ultima/videos/ep02_D-CHQSm1YwU..mp4"
Deleting original file data/raw/fabula_ultima/videos/ep02_D-CHQSm1YwU..f234.mp4 (pass -k to keep)
Deleting original file data/raw/fabula_ultima/videos/ep02_D-CHQSm1YwU..f616.mp4 (pass -k to keep)
✅ Vídeo salvo: ep02_D-CHQSm1YwU..%(ext)s

🔄 Fazendo down

[youtube] D-CHQSm1YwU: Downloading m3u8 information
[info] D-CHQSm1YwU: Downloading 1 format(s): 234
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1761
[download] Destination: data/raw/fabula_ultima/audio/ep02_D-CHQSm1YwU..mp4
[download] 100% of  149.53MiB in 00:01:56 at 1.29MiB/s                     
[ExtractAudio] Destination: data/raw/fabula_ultima/audio/ep02_D-CHQSm1YwU..wav
Deleting original file data/raw/fabula_ultima/audio/ep02_D-CHQSm1YwU..mp4 (pass -k to keep)
✅ Áudio salvo: ep02_D-CHQSm1YwU..wav

🔄 Salvando metadados...
✅ Metadados salvos: ep02_D-CHQSm1YwU..json

📊 RESUMO:
   📂 Campanha: fabula_ultima
   📍 Episódio: ep02 (detectado automaticamente)
   📹 Vídeo: ep02_D-CHQSm1YwU..%(ext)s
   🔊 Áudio: ep02_D-CHQSm1YwU..wav
   📋 Metadados: ep02_D-CHQSm1YwU..json

📁 Estrutura atual da campanha:
   📄 audio/ep01_f-tmb-Tcn_k..wav
   📄 audio/ep02_D-CHQSm1YwU..wav
   📄 metadata/ep01_f-tmb-Tcn_k..json
   📄 metadata/ep02_D-CHQSm1YwU..json
   📄 videos/ep01_f-tmb-Tcn_k.

In [ ]:
# 📦 Importações
from yt_dlp import YoutubeDL
from pathlib import Path
from datetime import datetime
import json
import re

def get_next_episode_number(campanha_dir: Path) -> str:
    """
    Verifica quantos episódios já existem e retorna o próximo número
    """
    video_dir = campanha_dir / "videos"
    
    if not video_dir.exists():
        return "ep01"  # Primeiro episódio
    
    # Buscar todos os arquivos que começam com "ep"
    existing_files = list(video_dir.glob("ep*"))
    
    if not existing_files:
        return "ep01"  # Nenhum episódio encontrado
    
    # Extrair números dos episódios existentes
    episode_numbers = []
    for file in existing_files:
        # Procurar padrão "ep" seguido de números
        match = re.search(r'ep(\d+)', file.name)
        if match:
            episode_numbers.append(int(match.group(1)))
    
    if not episode_numbers:
        return "ep01"
    
    # Próximo número
    next_number = max(episode_numbers) + 1
    return f"ep{next_number:02d}"  # Formato ep01, ep02, etc.

def get_campaign_from_url_or_title(url: str, title: str = "") -> str:
    """
    Tenta identificar a campanha pela URL ou título
    """
    # Dicionário de campanhas conhecidas
    known_campaigns = {
        "fabula ultima": "fabula_ultima",
        "#fabulaultima": "fabula_ultima",
        "ordem paranormal": "ordem_paranormal", 
        "critical role": "critical_role",
        "jambô": "jambo_editora",
        "cellbit": "ordem_paranormal"
    }
    
    # Verificar URL primeiro
    url_lower = url.lower()
    for keyword, campaign in known_campaigns.items():
        if keyword in url_lower:
            return campaign
    
    # Verificar título se fornecido
    if title:
        title_lower = title.lower()
        for keyword, campaign in known_campaigns.items():
            if keyword in title_lower:
                return campaign
    
    # Fallback: perguntar ao usuário
    return None

def ask_user_for_campaign() -> str:
    """
    Pergunta ao usuário qual campanha quando não consegue detectar automaticamente
    """
    print("\n🤔 Não consegui identificar a campanha automaticamente.")
    print("Campanhas disponíveis:")
    print("  1. fabula_ultima")
    print("  2. ordem_paranormal") 
    print("  3. critical_role")
    print("  4. nova_campanha (criar nova)")
    
    while True:
        choice = input("Digite o número ou nome da campanha: ").strip()
        
        if choice == "1":
            return "fabula_ultima"
        elif choice == "2":
            return "ordem_paranormal"
        elif choice == "3":
            return "critical_role"
        elif choice == "4":
            new_name = input("Nome da nova campanha (sem espaços): ").strip()
            return new_name.replace(" ", "_").lower()
        elif choice.replace(" ", "_").lower():
            return choice.replace(" ", "_").lower()
        else:
            print("❌ Opção inválida. Tente novamente.")

# 🛠️ Configuração
URL = "https://www.youtube.com/watch?v=f-tmb-Tcn_k"

# Data de extração
DATA_EXTRACAO = datetime.now().strftime("%Y%m%d")

print("🔍 Analisando vídeo para detectar campanha e episódio...")

# Primeiro, extrair informações do vídeo
try:
    with YoutubeDL({'quiet': True}) as ydl:
        info = ydl.extract_info(URL, download=False)
    
    titulo = info.get('title', '')
    video_id = info.get('id', 'unknown')
    upload_date = info.get('upload_date', '00000000')
    
    print(f"📺 Título: {titulo}")
    
    # Tentar detectar campanha
    campanha = get_campaign_from_url_or_title(URL, titulo)
    
    if not campanha:
        campanha = ask_user_for_campaign()
    
    print(f"🎯 Campanha detectada: {campanha}")
    
except Exception as e:
    print(f"❌ Erro ao extrair informações: {e}")
    campanha = ask_user_for_campaign()
    video_id = 'unknown'
    upload_date = '00000000'
    titulo = 'Título desconhecido'

# Estrutura de pastas
BASE_DIR = Path("data/raw")
CAMPANHA_DIR = BASE_DIR / campanha

# Detectar próximo episódio automaticamente
episodio = get_next_episode_number(CAMPANHA_DIR)
print(f"📍 Próximo episódio: {episodio}")

# Criar estrutura de pastas
VIDEO_DIR = CAMPANHA_DIR / "videos"
AUDIO_DIR = CAMPANHA_DIR / "audio"
META_DIR = CAMPANHA_DIR / "metadata"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Função para criar nomes de arquivo (versão simplificada)
def create_filename(episodio: str, video_id: str, extension: str) -> str:
    def clean_filename(text: str) -> str:
        cleaned = re.sub(r'[^\w\-]', '', text)
        return cleaned[:15]  # Limita tamanho
    
    episodio_clean = clean_filename(episodio)
    video_id_clean = clean_filename(video_id)
    
    # Remover pontos finais desnecessários
    filename = f"{episodio_clean}_{video_id_clean}"
    if extension:
        filename += f".{extension}"
    
    return filename

# Nome base do arquivo (simplificado)
base_filename = create_filename(
    episodio=episodio,
    video_id=video_id,
    extension=""
)

print(f"📝 Nome do arquivo: {base_filename}")

# 🎬 Download do vídeo
print("\n🔄 Fazendo download do vídeo...")

video_filename = base_filename + ".%(ext)s"
video_path = VIDEO_DIR / video_filename

video_opts = {
    "format": "bestvideo+bestaudio/best",
    "outtmpl": str(video_path),
    "writeinfojson": False,  # Não gerar arquivo .info.json
}

try:
    with YoutubeDL(video_opts) as ydl:
        ydl.download([URL])
    print(f"✅ Vídeo salvo: {video_filename}")
    
except Exception as e:
    print(f"❌ Erro no download do vídeo: {e}")

# 🔊 Download do áudio
print("\n🔄 Fazendo download do áudio...")

audio_filename = base_filename + ".wav"
audio_path = AUDIO_DIR / audio_filename

audio_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(AUDIO_DIR / (base_filename + ".%(ext)s")),
    "postprocessors": [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '192',
    }],
    "prefer_ffmpeg": True,
}

try:
    with YoutubeDL(audio_opts) as ydl:
        ydl.download([URL])
    print(f"✅ Áudio salvo: {audio_filename}")
    
except Exception as e:
    print(f"❌ Erro no download do áudio: {e}")

# 📊 Salvar metadados
print("\n🔄 Salvando metadados...")

metadata_filename = base_filename + ".json"
metadata_path = META_DIR / metadata_filename

metadata = {
    "campanha": campanha,
    "episodio": episodio,
    "data_extracao": DATA_EXTRACAO,
    "titulo": titulo,
    "url_original": URL,
    "video_id": video_id,
    "upload_date": upload_date,
    "deteccao_automatica": True,
    "arquivos_gerados": {
        "video": video_filename,
        "audio": audio_filename,
        "metadata": metadata_filename
    }
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadados salvos: {metadata_filename}")

# 📋 Relatório final
print(f"\n📊 RESUMO:")
print(f"   📂 Campanha: {campanha}")
print(f"   📍 Episódio: {episodio} (detectado automaticamente)")
print(f"   📹 Vídeo: {video_filename}")
print(f"   🔊 Áudio: {audio_filename}")
print(f"   📋 Metadados: {metadata_filename}")

# Mostrar estrutura de pastas atual
# Retirar!
print(f"\n📁 Estrutura atual da campanha:")
for item in sorted(CAMPANHA_DIR.rglob("*")):
    if item.is_file():
        relative_path = item.relative_to(CAMPANHA_DIR)
        print(f"   📄 {relative_path}")

print("\n🎉 Download concluído com numeração automática!")

🔍 Analisando vídeo para detectar campanha e episódio...


📺 Título: #FabulaUltima: OS CHAMPIÕES — Episódio #01 | Campanha Oficial Brasileira
🎯 Campanha detectada: fabula_ultima
📍 Próximo episódio: ep03
📝 Nome do arquivo: ep03_f-tmb-Tcn_k

🔄 Fazendo download do vídeo...
[youtube] Extracting URL: https://www.youtube.com/watch?v=f-tmb-Tcn_k
[youtube] f-tmb-Tcn_k: Downloading webpage
[youtube] f-tmb-Tcn_k: Downloading ios player API JSON
[youtube] f-tmb-Tcn_k: Downloading mweb player API JSON
[youtube] f-tmb-Tcn_k: Downloading player 6e20d3a8


[youtube] f-tmb-Tcn_k: Downloading m3u8 information
[info] f-tmb-Tcn_k: Downloading 1 format(s): 616+234
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1676
[download] Destination: data/raw/fabula_ultima/videos/ep03_f-tmb-Tcn_k.f616.mp4
[download] 100% of    1.67GiB in 00:02:04 at 13.81MiB/s                    
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1676
[download] Destination: data/raw/fabula_ultima/videos/ep03_f-tmb-Tcn_k.f234.mp4
[download] 100% of  144.01MiB in 00:01:10 at 2.06MiB/s                     
[Merger] Merging formats into "data/raw/fabula_ultima/videos/ep03_f-tmb-Tcn_k.mp4"
Deleting original file data/raw/fabula_ultima/videos/ep03_f-tmb-Tcn_k.f234.mp4 (pass -k to keep)
Deleting original file data/raw/fabula_ultima/videos/ep03_f-tmb-Tcn_k.f616.mp4 (pass -k to keep)
✅ Vídeo salvo: ep03_f-tmb-Tcn_k.%(ext)s

🔄 Fazendo download do áudio...
[youtube] Extracting URL: https://www.youtube.com/watch?v=f-tmb-Tcn_k
[youtube] f-tmb-Tcn_

[youtube] f-tmb-Tcn_k: Downloading m3u8 information
[info] f-tmb-Tcn_k: Downloading 1 format(s): 234
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1676
[download] Destination: data/raw/fabula_ultima/audio/ep03_f-tmb-Tcn_k.mp4
[download] 100% of  144.01MiB in 00:01:00 at 2.37MiB/s                     
[ExtractAudio] Destination: data/raw/fabula_ultima/audio/ep03_f-tmb-Tcn_k.wav
Deleting original file data/raw/fabula_ultima/audio/ep03_f-tmb-Tcn_k.mp4 (pass -k to keep)
✅ Áudio salvo: ep03_f-tmb-Tcn_k.wav

🔄 Salvando metadados...
✅ Metadados salvos: ep03_f-tmb-Tcn_k.json

📊 RESUMO:
   📂 Campanha: fabula_ultima
   📍 Episódio: ep03 (detectado automaticamente)
   📹 Vídeo: ep03_f-tmb-Tcn_k.%(ext)s
   🔊 Áudio: ep03_f-tmb-Tcn_k.wav
   📋 Metadados: ep03_f-tmb-Tcn_k.json

📁 Estrutura atual da campanha:
   📄 audio/ep01_f-tmb-Tcn_k..wav
   📄 audio/ep02_D-CHQSm1YwU..wav
   📄 audio/ep03_f-tmb-Tcn_k.wav
   📄 metadata/ep01_f-tmb-Tcn_k..json
   📄 metadata/ep02_D-CHQSm1YwU..json
   📄 

In [ ]:
# 📦 Importações
from yt_dlp import YoutubeDL
from pathlib import Path
from datetime import datetime
import json
import re

def get_next_episode_number(campanha_dir: Path) -> str:
    """
    Verifica quantos episódios já existem e retorna o próximo número
    """
    video_dir = campanha_dir / "videos"
    
    if not video_dir.exists():
        return "ep01"  # Primeiro episódio
    
    # Buscar todos os arquivos que começam com "ep"
    existing_files = list(video_dir.glob("ep*"))
    
    if not existing_files:
        return "ep01"  # Nenhum episódio encontrado
    
    # Extrair números dos episódios existentes
    episode_numbers = []
    for file in existing_files:
        # Procurar padrão "ep" seguido de números
        match = re.search(r'ep(\d+)', file.name)
        if match:
            episode_numbers.append(int(match.group(1)))
    
    if not episode_numbers:
        return "ep01"
    
    # Próximo número
    next_number = max(episode_numbers) + 1
    return f"ep{next_number:02d}"  # Formato ep01, ep02, etc.

def get_campaign_from_url_or_title(url: str, title: str = "") -> str:
    """
    Tenta identificar a campanha pela URL ou título
    """
    # Dicionário de campanhas conhecidas
    known_campaigns = {
        "fabula ultima": "fabula_ultima",
        "#fabulaultima": "fabula_ultima",
        "ordem paranormal": "ordem_paranormal", 
        "critical role": "critical_role",
        "jambô": "jambo_editora",
        "cellbit": "ordem_paranormal"
    }
    
    # Verificar URL primeiro
    url_lower = url.lower()
    for keyword, campaign in known_campaigns.items():
        if keyword in url_lower:
            return campaign
    
    # Verificar título se fornecido
    if title:
        title_lower = title.lower()
        for keyword, campaign in known_campaigns.items():
            if keyword in title_lower:
                return campaign
    
    # Fallback: perguntar ao usuário
    return None

def ask_user_for_campaign() -> str:
    """
    Pergunta ao usuário qual campanha quando não consegue detectar automaticamente
    """
    print("\n🤔 Não consegui identificar a campanha automaticamente.")
    print("Campanhas disponíveis:")
    print("  1. fabula_ultima")
    print("  2. ordem_paranormal") 
    print("  3. critical_role")
    print("  4. nova_campanha (criar nova)")
    
    while True:
        choice = input("Digite o número ou nome da campanha: ").strip()
        
        if choice == "1":
            return "fabula_ultima"
        elif choice == "2":
            return "ordem_paranormal"
        elif choice == "3":
            return "critical_role"
        elif choice == "4":
            new_name = input("Nome da nova campanha (sem espaços): ").strip()
            return new_name.replace(" ", "_").lower()
        elif choice.replace(" ", "_").lower():
            return choice.replace(" ", "_").lower()
        else:
            print("❌ Opção inválida. Tente novamente.")

# 🛠️ Configuração
URL = "https://www.youtube.com/watch?v=f-tmb-Tcn_k"

# Data de extração
DATA_EXTRACAO = datetime.now().strftime("%Y%m%d")

print("🔍 Analisando vídeo para detectar campanha e episódio...")

# Primeiro, extrair informações do vídeo
try:
    with YoutubeDL({'quiet': True}) as ydl:
        info = ydl.extract_info(URL, download=False)
    
    titulo = info.get('title', '')
    video_id = info.get('id', 'unknown')
    upload_date = info.get('upload_date', '00000000')
    
    print(f"📺 Título: {titulo}")
    
    # Tentar detectar campanha
    campanha = get_campaign_from_url_or_title(URL, titulo)
    
    if not campanha:
        campanha = ask_user_for_campaign()
    
    print(f"🎯 Campanha detectada: {campanha}")
    
except Exception as e:
    print(f"❌ Erro ao extrair informações: {e}")
    campanha = ask_user_for_campaign()
    video_id = 'unknown'
    upload_date = '00000000'
    titulo = 'Título desconhecido'

# Estrutura de pastas
BASE_DIR = Path("data/raw")
CAMPANHA_DIR = BASE_DIR / campanha

# Detectar próximo episódio automaticamente
episodio = get_next_episode_number(CAMPANHA_DIR)
print(f"📍 Próximo episódio: {episodio}")

# Criar estrutura de pastas
VIDEO_DIR = CAMPANHA_DIR / "videos"
AUDIO_DIR = CAMPANHA_DIR / "audio"
META_DIR = CAMPANHA_DIR / "metadata"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

# Função para criar nomes de arquivo (versão simplificada)
def create_filename(episodio: str, video_id: str, extension: str) -> str:
    def clean_filename(text: str) -> str:
        cleaned = re.sub(r'[^\w\-]', '', text)
        return cleaned[:15]  # Limita tamanho
    
    episodio_clean = clean_filename(episodio)
    video_id_clean = clean_filename(video_id)
    
    # Remover pontos finais desnecessários
    filename = f"{episodio_clean}_{video_id_clean}"
    if extension:
        filename += f".{extension}"
    
    return filename

# Nome base do arquivo (simplificado)
base_filename = create_filename(
    episodio=episodio,
    video_id=video_id,
    extension=""
)

print(f"📝 Nome do arquivo: {base_filename}")

# 🎬 Download do vídeo
print("\n🔄 Fazendo download do vídeo...")

video_filename = base_filename + ".%(ext)s"
video_path = VIDEO_DIR / video_filename

video_opts = {
    "format": "bestvideo+bestaudio/best",
    "outtmpl": str(video_path),
    "writeinfojson": False,  # Não gerar arquivo .info.json
}

try:
    with YoutubeDL(video_opts) as ydl:
        ydl.download([URL])
    print(f"✅ Vídeo salvo: {video_filename}")
    
except Exception as e:
    print(f"❌ Erro no download do vídeo: {e}")

# 🔊 Download do áudio
print("\n🔄 Fazendo download do áudio...")

audio_filename = base_filename + ".wav"
audio_path = AUDIO_DIR / audio_filename

audio_opts = {
    "format": "bestaudio/best",
    "outtmpl": str(AUDIO_DIR / (base_filename + ".%(ext)s")),
    "postprocessors": [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
        'preferredquality': '192',
    }],
    "prefer_ffmpeg": True,
}

try:
    with YoutubeDL(audio_opts) as ydl:
        ydl.download([URL])
    print(f"✅ Áudio salvo: {audio_filename}")
    
except Exception as e:
    print(f"❌ Erro no download do áudio: {e}")

# 📊 Salvar metadados
print("\n🔄 Salvando metadados...")

metadata_filename = base_filename + ".json"
metadata_path = META_DIR / metadata_filename

metadata = {
    "campanha": campanha,
    "episodio": episodio,
    "data_extracao": DATA_EXTRACAO,
    "titulo": titulo,
    "url_original": URL,
    "video_id": video_id,
    "upload_date": upload_date,
    "deteccao_automatica": True,
    "arquivos_gerados": {
        "video": video_filename,
        "audio": audio_filename,
        "metadata": metadata_filename
    }
}

with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadados salvos: {metadata_filename}")

# 📋 Relatório final
print(f"\n📊 RESUMO:")
print(f"   📂 Campanha: {campanha}")
print(f"   📍 Episódio: {episodio} (detectado automaticamente)")
print(f"   📹 Vídeo: {video_filename}")
print(f"   🔊 Áudio: {audio_filename}")
print(f"   📋 Metadados: {metadata_filename}")